# Isolated Level 0 — multi-seed diagnostics

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT', '/tmp/nanogpt-level0-gpt2/results'))
rows = []
summary_rows = []
for path in sorted(ROOT.glob('*_seed_*/metrics.csv')):
    run = path.parent
    completion_path = run / 'run_complete.json'
    selected_path = run / 'selected_checkpoint_metrics.json'
    if not completion_path.is_file() or not selected_path.is_file():
        continue
    optimizer, seed_text = run.name.rsplit('_seed_', 1)
    frame = pd.read_csv(path)
    frame['optimizer'] = optimizer
    frame['seed'] = int(seed_text)
    rows.append(frame)
    selected = json.loads(selected_path.read_text())
    completion = json.loads(completion_path.read_text())
    summary_rows.append({
        'optimizer': optimizer,
        'seed': int(seed_text),
        'selected_step': selected['selected_step'],
        'selected_validation_loss': selected['validation_loss'],
        'selected_test_loss': selected['test_loss'],
        'selected_test_perplexity': selected['test_perplexity'],
        'selected_test_accuracy_percent': 100 * selected['test_accuracy'],
        'final_test_loss': completion['final_test_loss'],
    })
if not rows:
    raise FileNotFoundError(f'No completed runs under {ROOT}')
all_metrics = pd.concat(rows, ignore_index=True)
summary = pd.DataFrame(summary_rows).sort_values(['optimizer', 'seed'])
summary

In [ ]:
def band_plot(column, ylabel):
    plt.figure(figsize=(10, 6))
    for optimizer, frame in all_metrics.groupby('optimizer'):
        aggregated = frame.groupby('tokens_seen')[column].agg(['mean', 'std']).reset_index()
        spread = aggregated['std'].fillna(0.0)
        line, = plt.plot(aggregated['tokens_seen'], aggregated['mean'], label=optimizer)
        plt.fill_between(
            aggregated['tokens_seen'],
            aggregated['mean'] - spread,
            aggregated['mean'] + spread,
            alpha=0.2,
            color=line.get_color(),
        )
    plt.xlabel('training tokens')
    plt.ylabel(ylabel)
    plt.title(f'{column}: mean ± one standard deviation')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

band_plot('val_loss', 'validation cross-entropy loss')
band_plot('val_perplexity', 'validation perplexity')
band_plot('val_accuracy', 'exact next-token accuracy (fraction)')
band_plot('val_generalization_gap', 'validation loss − training loss')

In [ ]:
ww_rows = []
for run in sorted(ROOT.glob('*_seed_*')):
    if '_seed_' not in run.name:
        continue
    optimizer, seed_text = run.name.rsplit('_seed_', 1)
    for path in sorted(run.glob('weightwatcher_step_*.csv')):
        frame = pd.read_csv(path)
        frame['optimizer'] = optimizer
        frame['seed'] = int(seed_text)
        ww_rows.append(frame)
if not ww_rows:
    print('No successful WeightWatcher measurements found.')
else:
    ww = pd.concat(ww_rows, ignore_index=True)
    ww['alpha'] = pd.to_numeric(ww.get('alpha'), errors='coerce')
    layer_column = next((name for name in ('source_layer', 'longname', 'name', 'layer_id') if name in ww), None)
    valid = ww[np.isfinite(ww['alpha'])].copy()
    for layer, layer_frame in valid.groupby(layer_column):
        plt.figure(figsize=(10, 5))
        for optimizer, frame in layer_frame.groupby('optimizer'):
            aggregated = frame.groupby('step')['alpha'].agg(['mean', 'std']).reset_index()
            spread = aggregated['std'].fillna(0.0)
            line, = plt.plot(aggregated['step'], aggregated['mean'], label=optimizer)
            plt.fill_between(
                aggregated['step'],
                aggregated['mean'] - spread,
                aggregated['mean'] + spread,
                alpha=0.2,
                color=line.get_color(),
            )
        plt.axhline(2.0, linestyle='--', linewidth=1)
        plt.xlabel('optimizer step')
        plt.ylabel('WeightWatcher alpha')
        plt.title(str(layer))
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()